# Main benchmark

Reproduces the main benchmark: Figure 2 and the paired permutation tests. The features of every model go through the same linear head and the same nested cross validation on all subjects of a dataset.

The notebook runs one dataset, set by `DATASET`: PPMI, NKI or HBN. With 16 CPU threads a dataset takes a few hours, mostly feature extraction. `sbatch slurm/main_benchmark.sh` runs the three datasets as a SLURM array, and `02_main_benchmark_results.ipynb` combines them.

Outputs:
- `outputs/main_benchmark/{DATASET}_folds.csv`: every metric for every model, task and outer fold
- `outputs/main_benchmark/{DATASET}_tests.csv`: the permutation tests of the dataset
- `outputs/main_benchmark/oof/`: out of fold predictions per subject
- `outputs/features/`: the extracted features and labels, reused by `04_head_timing.ipynb`

In [ ]:
import os

DATASET = os.environ.get("DATASET", "NKI")
N_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))

In [ ]:
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[var] = str(N_CPUS)

import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, mean_absolute_error

import cohorts
import encoders
import heads
from config import OUTPUT_DIR

torch.set_num_threads(N_CPUS)
torch.set_num_interop_threads(N_CPUS)

OUT = OUTPUT_DIR / "main_benchmark"
FEATURES = OUTPUT_DIR / "features"
(OUT / "oof").mkdir(parents=True, exist_ok=True)
FEATURES.mkdir(parents=True, exist_ok=True)
print(f"{DATASET}, {N_CPUS} threads")

## Cohort

A subject is kept when it has an affine TurboPrep volume, a CAT12 gray matter map, complete FreeSurfer features and the labels, as built by `cohorts.py`. BMI is evaluated on the subjects with 10 < BMI < 80, and PPMI has no BMI. In HBN, three subjects whose T1w images are byte identical to another subject's in the Reproducible Brain Charts release are excluded.

In [ ]:
cohort = cohorts.load(DATASET)
subjects = np.asarray(cohort.subjects)
TASKS = {"sex": (slice(None), cohort.sex), "age": (slice(None), cohort.age)}
if cohort.bmi is not None:
    TASKS["bmi"] = (cohort.bmi_idx, cohort.bmi)
pd.Series(cohorts.describe(cohort)).round(2)

## Features

| Model | Input | Input preparation | Feature |
|---|---|---|---|
| un-CNN | affine volume | resized to 91 × 109 × 91; robust scaling; intensity, rank filtered and Sobel magnitude channels | 9,664 |
| AnatCL Global, AnatCL Local | CAT12 gray matter map | resized to 121 × 128 × 121 | mean over the five fold models, 512 |
| BrainIAC | affine volume | resized to 96 × 96 × 96, standardized | first output token, 768 |
| SwinBrain | affine volume | resized to 128 × 128 × 64, standardized, repeated over three channels | average pooled encoder10 output, 384 |
| 3D-Neuro-SimCLR | affine volume | axes reordered, center cropped or zero padded to 150 × 192 × 192, standardized | 512 |
| FreeSurfer Schaefer | FreeSurfer v6.0.1, NKI and HBN | surface area and thickness of 400 parcels | 800 |
| FreeSurfer aparc | FreeSurfer v6.0.1, NKI and HBN | thickness and surface area of 68 regions | 136 |
| FreeSurfer | FreeSurfer v7.3.2, PPMI | aparc thickness and surface area, plus mean thickness and white surface area per hemisphere | 140 |

Every encoder is frozen and runs one forward pass per image on CPU; the raw features are used without further transformation. 3D-Neuro-SimCLR and BrainIAC are not evaluated on PPMI, which is part of their pretraining data. The un-CNN features are those of `uncnn.extract_features(files, batch_size=1, device="cpu")`.

In [ ]:
MODELS = ["Untrained_CNN", "AnatCL_Global", "AnatCL_Local", "BrainIAC", "SwinBrain", "3D-Neuro-SimCLR"]
if DATASET == "PPMI":
    MODELS = [m for m in MODELS if m not in ("BrainIAC", "3D-Neuro-SimCLR")]

features = {}
for name in MODELS:
    t0 = time.time()
    files = cohort.cat12 if encoders.ENCODERS[name][3] == "cat12" else cohort.affine
    features[name] = encoders.extract_features(name, files)
    print(f"{name:<16} {features[name].shape}  {(time.time() - t0) / 60:.1f} min")
features.update(cohort.freesurfer)

for name, X in features.items():
    np.save(FEATURES / f"{DATASET}_{name}.npy", X)
(FEATURES / f"{DATASET}_subjects.txt").write_text("\n".join(cohort.subjects) + "\n")
np.savez(FEATURES / f"{DATASET}_labels.npz", sex=cohort.sex, age=cohort.age,
         bmi_idx=np.array([] if cohort.bmi_idx is None else cohort.bmi_idx, dtype=int),
         bmi=np.array([] if cohort.bmi is None else cohort.bmi, dtype=float))

## Linear head and nested cross validation

Every feature set goes through the same head, defined in `heads.py`. Features are standardized with the mean and standard deviation of the training fold and passed to a linear model with an unpenalized bias:

- age and BMI: ridge regression minimizing the sum of squared errors plus $\alpha\|w\|_2^2$, solved in closed form, with $\alpha$ from 30 log spaced values between $10^{-2}$ and $10^{5}$;
- sex: $L_2$ regularized logistic regression minimizing $C\sum_i \mathrm{BCE}_i + \frac{1}{2}\|w\|_2^2$ with L-BFGS, at most 2,000 iterations, with $C$ from 30 log spaced values between $10^{-5}$ and $10^{5}$.

The outer loop has 10 folds, StratifiedKFold for sex and KFold for age and BMI, shuffled with seed 0. Folds are built once per task and shared by all models, so the results are paired; BMI folds contain only the subjects with BMI. Within each outer training set, $\alpha$ or $C$ is selected by inner fivefold cross validation, by lowest mean squared error for regression and highest accuracy for sex; the head is then refit on the whole outer training set and scored on the held out fold. The chance baseline, `Dummy`, predicts the training fold majority class for sex and the training fold mean for age and BMI.

Sex is scored by balanced accuracy and accuracy, age and BMI by mean absolute error.

In [ ]:
folds = {task: heads.make_folds(y, task) for task, (_, y) in TASKS.items()}
rows, cv_means, oof = [], {}, {}


def cross_validate(name, X, y, task, subj):
    is_clf = task == "sex"
    n = len(y)
    fold_id, y_pred, logit = np.empty(n, dtype=int), np.empty(n), np.empty(n)
    metrics = {"balanced_accuracy": [], "accuracy": [], "C": []} if is_clf else {"mae": [], "alpha": []}
    n_nonconv = 0

    for k, (tr, te) in enumerate(folds[task]):
        if name == "Dummy":
            # Constant prediction; for sex +1 or -1, so that out >= 0 gives the majority class.
            const = (1.0 if np.mean(y[tr]) >= 0.5 else -1.0) if is_clf else float(np.mean(y[tr]))
            out = np.full(len(te), const)
        else:
            model, lam, nc = heads.select_and_fit(X[tr], y[tr], task)
            n_nonconv += nc
            out = heads.predict_head(model, X[te])
            metrics["C" if is_clf else "alpha"].append(lam)
        fold_id[te] = k
        if is_clf:
            pred = (out >= 0).astype(int)
            logit[te], y_pred[te] = out, pred
            metrics["balanced_accuracy"].append(balanced_accuracy_score(y[te], pred))
            metrics["accuracy"].append(accuracy_score(y[te], pred))
        else:
            y_pred[te] = out
            metrics["mae"].append(mean_absolute_error(y[te], out))

    for metric, values in metrics.items():
        rows.extend({"dataset": DATASET, "model": name, "task": task, "metric": metric, "fold": fold,
                     "value": v, "head": "linear_sklearn"} for fold, v in enumerate(values))

    if is_clf:
        y_pred = y_pred.astype(int)
    pred_df = pd.DataFrame({"subject": subj, "fold": fold_id, "y_true": y, "y_pred": y_pred})
    if is_clf:
        pred_df["logit"] = logit
    pred_df.to_csv(OUT / "oof" / f"{DATASET}_{name}_{task}.csv", index=False)

    primary = "balanced_accuracy" if is_clf else "mae"
    cv_means.setdefault(task, {})[name] = float(np.mean(metrics[primary]))
    oof.setdefault(task, {})[name] = (np.asarray(y), y_pred)
    if n_nonconv:
        print(f"  [warn] {DATASET} {name} {task}: {n_nonconv} fits raised ConvergenceWarning")
    return metrics[primary]


features["Dummy"] = np.zeros((len(subjects), 1))
for name, X in features.items():
    scores = {task: cross_validate(name, X[idx], y, task, subjects[idx]) for task, (idx, y) in TASKS.items()}
    print(f"  {name:<25}" + "   ".join(f"{t}: {np.mean(s):.3f}+/-{np.std(s):.3f}" for t, s in scores.items()))

folds_df = pd.DataFrame(rows)
folds_df.to_csv(OUT / f"{DATASET}_folds.csv", index=False)

## Paired permutation test

For each task, the un-CNN is compared with the best other model by mean cross validated score; the chance baseline is not a candidate. The test pools the out of fold predictions and, in each of 10,000 permutations, swaps the two models' predictions of a subject with probability 0.5. The statistic is the difference in balanced accuracy or mean absolute error, two sided, with $p = (\text{count} + 1) / 10{,}001$. Bonferroni correction is over the 8 dataset and task comparisons of the paper, sex and age on PPMI and sex, age and BMI on NKI and HBN, and a difference is significant when its corrected $p < 0.05$. Because the competitor is the strongest model on the same results, the test is conservative for the un-CNN.

In [ ]:
N_PERMUTATIONS = 10000
N_TESTS = 8
FWER_ALPHA = 0.05


def permutation_test(y, pred_a, pred_b, metric_fn):
    obs = metric_fn(y, pred_a) - metric_fn(y, pred_b)
    rng = np.random.default_rng(0)
    count = 0
    for _ in range(N_PERMUTATIONS):
        flip = rng.random(len(y)) < 0.5
        a = np.where(flip, pred_b, pred_a)
        b = np.where(flip, pred_a, pred_b)
        # The tolerance keeps permutations that reproduce |obs| from being lost to rounding.
        if abs(metric_fn(y, a) - metric_fn(y, b)) >= abs(obs) - 1e-12:
            count += 1
    return obs, (count + 1) / (N_PERMUTATIONS + 1)


tests = []
for task, means in cv_means.items():
    is_clf = task == "sex"
    others = {m: v for m, v in means.items() if m not in ("Untrained_CNN", "Dummy")}
    competitor = (max if is_clf else min)(others, key=others.get)
    metric_fn = balanced_accuracy_score if is_clf else mean_absolute_error
    y, pred_u = oof[task]["Untrained_CNN"]
    y_c, pred_c = oof[task][competitor]
    assert np.array_equal(y, y_c)
    obs, p = permutation_test(y, pred_u, pred_c, metric_fn)
    tests.append({"dataset": DATASET, "task": task, "uncnn_metric": metric_fn(y, pred_u), "competitor": competitor,
                  "competitor_metric": metric_fn(y, pred_c), "observed_diff": obs, "p_raw": p,
                  "p_bonferroni": min(N_TESTS * p, 1.0), "significant": p < FWER_ALPHA / N_TESTS})

tests_df = pd.DataFrame(tests)
tests_df.to_csv(OUT / f"{DATASET}_tests.csv", index=False)
tests_df.round(4)

## Results

Mean and standard deviation across the 10 outer folds, the values shown in Figure 2.

In [ ]:
primary = folds_df[folds_df.metric.isin(["balanced_accuracy", "mae"])]
(primary.groupby(["model", "task"], sort=False)["value"]
 .agg(lambda v: f"{v.mean():.3f} ± {v.std(ddof=0):.3f}").unstack("task")[list(TASKS)])